<a href="https://colab.research.google.com/github/lee-doris/HealthCareAnalysis/blob/main/20260811.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install duckdb -q

# !：告訴 Colab 這一行要當作 Bash 命令（Linux 系統指令）執行，而不是 Python 程式碼。
# pip install duckdb：自動從 Python 套件庫下載並安裝 DuckDB。
# -q (quiet)：代表「安靜模式」，安裝時不會跳出一大堆下載與編譯日誌，保持畫面乾淨整潔。

# 每次重啟都需要執行一次：Colab 每次分配到的都是全新的雲端臨時虛擬機（Runtime）。當你重新連線或重啟 Runtime 時，之前安裝過的 DuckDB 就會消失，需要重新執行 !pip install duckdb -q。
# 同一階段只需執行一次：只要你還在同一個連線階段中，執行過一次 !pip install 後，後續即使新增好幾個程式碼區塊（Cell），都可以直接寫 import duckdb，不需要重複安裝。

import duckdb
import pandas as pd

url = "https://raw.githubusercontent.com/lee-doris/HealthCareAnalysis/refs/heads/main/enhanced_health_insurance_claims.csv"


In [7]:
#方法一

# DuckDB 直接讀取 URL 網址
result = duckdb.query(f"SELECT * FROM '{url}' LIMIT 5").df()

result.head(3)





# 沒有 f 的第一個寫法：查詢「Python 變數」
# result = duckdb.query("SELECT * FROM df LIMIT 5").df()
# 這裡的 df 是什麼：它是你在 Python 記憶體中建立好的 Pandas DataFrame 變數。
# 為什麼不需要 f：因為字串裡面的 df 不需要動態替換成別的文字，DuckDB 看到字串裡的 df，就會自動去 Python 環境中找名稱叫做 df 的變數。

# 有 f 的第二個寫法：查詢「動態網址/檔案路徑」
# result = duckdb.query(f"SELECT * FROM '{url}' LIMIT 5").df()
# 這裡的 f 是什麼：這是 Python 的 f-string（格式化字串） 語法。
# 為什麼一定要加 f：如果不加 f，SQL 語法會變成純文字 "SELECT * FROM '{url}' LIMIT 5"，DuckDB 會真的以為你要去找一個檔名叫作 {url} 的檔案，進而報錯。
# 加上 f 的效果：Python 會先將變數 {url} 替換成實際的網址，組裝成完整的 SQL 指令：
# SELECT * FROM 'https://raw.githubusercontent.com/.../claims.csv' LIMIT 5

# 不用 f：直接對記憶體中的 DataFrame 變數 下 SQL。
# 要用 f：要把 Python 變數儲存的 網址/路徑字串 帶入 SQL 語法中。

,ClaimID,PatientID,ProviderID,ClaimAmount,ClaimDate,DiagnosisCode,ProcedureCode,PatientAge,PatientGender,ProviderSpecialty,ClaimStatus,PatientIncome,PatientMaritalStatus,PatientEmploymentStatus,ProviderLocation,ClaimType,ClaimSubmissionMethod
0,10944daf-f7d5-4e1d-8216-72ffa609fe41,8552381d-7960-4f64-b190-b20b8ada00a1,4a4cb19c-4863-41cf-84b0-c2b21aace988,3807.95,2024-06-07,yy006,hd662,16,M,Cardiology,Pending,90279.43,Married,Retired,Jameshaven,Routine,Paper
1,fcbebb25-fc24-4c0f-a966-749edcf83fb1,327f43ad-e3bd-4473-a9ed-46483a0a156f,422e02dd-c1fd-43dd-8af4-0c3523f997b1,9512.07,2023-05-30,tD052,mH831,27,M,Pediatrics,Approved,130448.02,Single,Student,Beltrantown,Routine,Online
2,9e9983e7-9ea7-45f5-84d8-ce49ccd8a4a1,6f3acdf7-73aa-4afa-9c2e-b25b27bdb5b0,f7733b3f-0980-47b5-a7a0-ee390869355b,7346.74,2022-09-27,zx832,dg637,40,F,Cardiology,Pending,82417.54,Divorced,Employed,West Charlesport,Emergency,Online


In [5]:
#方法二
df = pd.read_csv(url)
result = duckdb.query("SELECT * FROM df LIMIT 5").df()
result.head(3)

,ClaimID,PatientID,ProviderID,ClaimAmount,ClaimDate,DiagnosisCode,ProcedureCode,PatientAge,PatientGender,ProviderSpecialty,ClaimStatus,PatientIncome,PatientMaritalStatus,PatientEmploymentStatus,ProviderLocation,ClaimType,ClaimSubmissionMethod
0,10944daf-f7d5-4e1d-8216-72ffa609fe41,8552381d-7960-4f64-b190-b20b8ada00a1,4a4cb19c-4863-41cf-84b0-c2b21aace988,3807.95,2024-06-07,yy006,hd662,16,M,Cardiology,Pending,90279.43,Married,Retired,Jameshaven,Routine,Paper
1,fcbebb25-fc24-4c0f-a966-749edcf83fb1,327f43ad-e3bd-4473-a9ed-46483a0a156f,422e02dd-c1fd-43dd-8af4-0c3523f997b1,9512.07,2023-05-30,tD052,mH831,27,M,Pediatrics,Approved,130448.02,Single,Student,Beltrantown,Routine,Online
2,9e9983e7-9ea7-45f5-84d8-ce49ccd8a4a1,6f3acdf7-73aa-4afa-9c2e-b25b27bdb5b0,f7733b3f-0980-47b5-a7a0-ee390869355b,7346.74,2022-09-27,zx832,dg637,40,F,Cardiology,Pending,82417.54,Divorced,Employed,West Charlesport,Emergency,Online


In [9]:
# 💡 在 Python 中用 DuckDB 一鍵執行
# 把上述 SQL 語法透過 DuckDB 轉成 DataFrame 輸出

import duckdb

# 1. 交叉比例表
pct_table = duckdb.query("""
    SELECT
        PatientMaritalStatus,
        ROUND(AVG(CASE WHEN ClaimStatus = 'Approved' THEN 1.0 ELSE 0 END) * 100, 2) AS Approved_Pct,
        ROUND(AVG(CASE WHEN ClaimStatus = 'Denied' THEN 1.0 ELSE 0 END) * 100, 2) AS Denied_Pct,
        ROUND(AVG(CASE WHEN ClaimStatus = 'Pending' THEN 1.0 ELSE 0 END) * 100, 2) AS Pending_Pct
    FROM df
    GROUP BY PatientMaritalStatus
""").df()

# 2. 平均與中位數金額表
amount_table = duckdb.query("""
    SELECT
        PatientMaritalStatus,
        ROUND(AVG(ClaimAmount), 2) AS Avg_Amount,
        ROUND(MEDIAN(ClaimAmount), 2) AS Median_Amount,
        COUNT(*) AS Total_Count
    FROM df
    GROUP BY PatientMaritalStatus
""").df()

print("--- 婚姻狀況與理賠狀態比例 (%) ---")
print(pct_table)

print("\n--- 婚姻狀況平均與中位數理賠金額 ---")
print(amount_table)


--- 婚姻狀況與理賠狀態比例 (%) ---
  PatientMaritalStatus  Approved_Pct  Denied_Pct  Pending_Pct
0              Married         32.85       35.31        31.84
1             Divorced         34.88       34.15        30.97
2              Widowed         34.69       30.43        34.87
3               Single         32.91       34.46        32.63

--- 婚姻狀況平均與中位數理賠金額 ---
  PatientMaritalStatus  Avg_Amount  Median_Amount  Total_Count
0               Single     4952.27        4971.95         1091
1              Married     4971.02        5062.63         1181
2             Divorced     5093.71        5123.38         1101
3              Widowed     5041.74        5071.73         1127
